### 字符编码演变顺序：ASCII -> Unicode -> UTF-32 -> UTF-8

<font size=2>

**ASCII**：占用1字节（8bit），但仅使用低7bit，取值范围为 0~127，只能表示英文字母、数字和基础标点符号，无法表示中文及其他国家文字。

**Unicode**：是一套全球统一的字符码点标准，为世界所有文字、符号分配唯一编号。Unicode 最大码点仅需要21位，使用4字节（32bit）的空间完全足够容纳所有字符，余量极大。

**UTF-32**：是 Unicode 的定长存储方式，无论字符大小，统一固定使用4字节存储一个码点。优点是读取规则简单、可按固定长度并行读取，解析速度快；缺点是空间浪费严重，简单字符也需要占用4字节存储空间。

**UTF-8**：是优化后的变长 Unicode 编码，字符占用字节数为1~4字节。有效解决了 UTF-32 空间浪费的问题，兼容 ASCII 编码；但由于长度不固定，解析时需要动态判断字节边界，解析效率略低于 UTF-32。

##### UTF-8 变长编码规则（标准模板）
Unicode 码点 **U+XXXX 为十六进制**，UTF-8 根据码点大小自动选择 1～4 字节存储。
x 代表填入字符的有效二进制位，固定前缀用于电脑识别字节长度。

| 十六进制码点区间    | 字节数  | UTF-8 二进制模板                     | 对应十进制范围   | 适用字符 |
|--------------------|--------|-------------------------------------|-----------------|----------|
| U+0000 ~ U+007F    | 1      | 0xxxxxxx                            | 0 ~ 127         | 英文、数字、符号（兼容ASCII） |
| U+0080 ~ U+07FF    | 2      | 110xxxxx 10xxxxxx                   | 128 ~ 2047      | 拼音、小语种文字 |
| U+0800 ~ U+FFFF    | 3      | 1110xxxx 10xxxxxx 10xxxxxx          | 2048 ~ 65535    | 绝大多数中文、日韩汉字 |
| U+10000 ~ U+10FFFF | 4      | 11110xxx 10xxxxxx 10xxxxxx 10xxxxxx | 65536 ~ 1114111 | 生僻字、Emoji、古汉字 |

##### 核心规则
1. 单字节首位为 0：和 ASCII 完全兼容。
2. 多字节字符：首字节告诉计算机总字节数，**后续所有字节固定以 10 开头**。
3. 最大码点 U+10FFFF，总字符数 1114112 个，4 字节完全够用。
</font>

In [ ]:
'''
汉字 UTF-8 编码完整转换流程拆解：
获取字符 Unicode 十进制码点，转为十六进制、二进制，得到字符原始二进制值
根据 Unicode 数值区间匹配 UTF-8 编码模板，拆分填充多段 8 位二进制字节
每一段二进制字节单独换算为十进制数字，就是 UTF-8 字节的十进制表现形式
以汉字「牛」为例：中文汉字在 UTF-8 下固定占用 3 个字节，对应 3 个 token。
Transformer 自注意力计算复杂度为 O (N²)，N 代表输入 token 数量；
单汉字拆分出 3 个 token 会增大算力开销，因此可通过字符级预编码、字表合并等方式压缩为单个 token，降低计算量。常用的就是BPE（Byte Pair Encoding）和 SentencePiece 等分词算法。
'''
print("牛") # 牛
print(ord("牛")) # 29305 unicode码点（10进制）
print(hex(ord("牛"))) # 0x72ce （16进制）
print(bin(ord("牛"))) # 0b1110010111001110 （2进制）
print('牛'.encode('utf-8')) # b'\xe7\x89\x9b' （utf-8编码）
print(f"[{' '.join(str(b) for b in '牛'.encode('utf-8'))}]") # [231 137 155] （utf-8编码的每个字节转成10进制）

# 表示的都是一样的，只是用数字看着更方便一些
print(b'\xe5\xa4\xa7'.decode("utf-8")) # 大
print(bytes([229, 164, 167]).decode("utf-8")) # 大 bytes([229, 164, 167] = b'\xe5\xa4\xa7'


牛
29275
0x725b
0b111001001011011
b'\xe7\x89\x9b'
[231 137 155]


### 预分词（Pre-tokenization）

<font size=2>
<!-- 基础定义 -->
预分词是前置粗分割工具，规则简单；核心约束：BPE后续合并只能在粗分割片段内部执行，不允许跨分割边界生成token。

<!-- 无预分词带来的语义问题 -->
案例1（中文）：句子「今天-天气不错」
不做预分词直接拆分：["今","天","-天","气"，"不错"]，完整短语被割裂；
先空格预分割得到["今天","-","天气不错"]后再分词，["今天","-","天气","不错"]短语边界保留，结果更符合语义。

案例2（英文）：句子「Hello Word」
无预分词风险：BPE可能跨单词拼接出 'oW' 这类无意义跨词组合；
正则预分词先切分独立单元["Hello","Word"]。

<!-- 预分词核心作用 -->
**强制合并仅在单个单词内部发生，杜绝跨词错误组合**
1. 规避BPE生成跨越语义边界的无效字符组合
2. 保全单词、短语的语义完整性
3. 减少无意义token，让模型更容易学习语言语义

<!-- 执行顺序强制规范 -->
**统计BPE字节对频率、执行字符合并操作前，必须先完成预分词处理**
</font>

In [ ]:
'''
<!-- 整体功能说明 -->
GPT-2 原生预分词（Pre-tokenization）正则模板，作用是粗分割文本，划定BPE不能跨越的语义边界，避免跨词产生无意义token。
依赖第三方 regex 库（支持Unicode正则属性 \p{}，Python内置re不支持）。
依赖安装：pip install regex
'''
import regex as re

Gpt2_pattern = re.compile(r"'s|'t|'re|'ve|'m|'ll|'d| ?[\p{L}\p{M}]+|[\p{N}]+|[\p{P}]+|[\p{S}]+|[\p{Z}]+")

text_1 = "今天-天气不错"
text_2 = "Hello World!"

print("text1预分词结果：", Gpt2_pattern.findall(text_1)) # ['今天', '-', '天气不错']
print("text2预分词结果：", Gpt2_pattern.findall(text_2)) # ['Hello', ' World', '!'] 空格会被保留，需要strip()去掉后再考虑预分词

# 预分词完成粗分割后，仅在单个预分词片段内部提取2-gram（连续两个字节构成的字节对）
# 1. 统计全部片段内所有字节对的出现频率；
# 2. 将全局频次最高的一组字节对合并为全新token；
# 3. 重复迭代两两合并操作，直至词表大小达到预设上限。
# 字节对提取、合并操作均不能跨预分词边界执行，杜绝拼接不同语义单元的字符，避免生成无实际语义的token。
# BPE名称中Pair代表成对，算法仅支持2元组合，不会一次性合并3个及以上字节，不属于通用n-gram。

text1预分词结果： ['今天', '-', '天气不错']
text2预分词结果： ['Hello', ' World', '!']


In [ ]:
# UTF-8是Unicode的一种表示方式，4Byte就可以表示全部的内容了，相当于不管该字符转换成UTF-8占几个字节，实际上每个字节都可以用[0~255]的范围表示。
# 后续词表的扩展就可以从256开始了。

print(list("大模型".encode('utf-8'))) # [229, 164, 167, 230, 168, 161, 229, 158, 139]
print("每个字节都在[0, 255]范围内" if all(x < 256 for x in "大模型".encode('utf-8')) else "有字节超出了[0, 255]范围") # 每个字节都在[0, 255]范围内

print(list("I can do it".encode('utf-8'))) # [73, 32, 99, 97, 110, 32, 100, 111, 32, 105, 116]
print("每个字节都在[0, 255]范围内" if all(x < 256 for x in "I can do it".encode('utf-8')) else "有字节超出了[0, 255]范围") # 每个字节都在[0, 255]范围内

print("====================================================================================")

# 假设每一个字符的字节都直接作为一个token，以"大模型"为例子，则会输入9个token，计算复杂度为O(9²)=81。如果输入再长一些，复杂度会爆炸，所以要想办法进行压缩
# 压缩的方式就是BPE（Byte Pair Encoding）和 SentencePiece 等分词算法，合并成更少的token，从而降低计算复杂度。
# 例如："大模型"[228, 184, 173, 230, 156, 172]，这其中每一个数字都代表一个vector。
'''
<!-- [228, 184, 173, 230, 156, 172] -->
tensor([[-1.9461, -2.5019,  1.6047, -0.2060],
        [-0.2072,  0.3835, -1.1414, -0.8583],
        [ 0.0890,  0.2215, -1.6362,  0.7986],
        [-1.0826, -0.5816, -0.1612, -0.8076],
        [-0.1428,  0.8030,  0.4615, -1.6574],
        [-0.0936, -0.1915,  0.2088, -0.8390],
        [-1.9461, -2.5019,  1.6047, -0.2060],
        [-0.8711,  0.5696,  1.3668,  1.2347],
        [ 0.0842, -0.3537, -0.0246, -0.7429]], grad_fn=<EmbeddingBackward0>)
'''
import torch
import torch.nn as nn

X = nn.Embedding(256, 4) # 256个token，每个token对应一个4维的向量
idx = torch.tensor([229, 164, 167, 230, 168, 161, 229, 158, 139]) # "大模型"的每个数字都是对应的vector的索引，转成Embedding可以识别的类型
print("随机初始化的，所以每一次的值都不同:\n" + str(X(idx))) # 将大模型中的每个数字的vector表示出来


[229, 164, 167, 230, 168, 161, 229, 158, 139]
每个字节都在[0, 255]范围内
[73, 32, 99, 97, 110, 32, 100, 111, 32, 105, 116]
每个字节都在[0, 255]范围内
随机初始化的，所以每一次的值都不同:
tensor([[-1.9461, -2.5019,  1.6047, -0.2060],
        [-0.2072,  0.3835, -1.1414, -0.8583],
        [ 0.0890,  0.2215, -1.6362,  0.7986],
        [-1.0826, -0.5816, -0.1612, -0.8076],
        [-0.1428,  0.8030,  0.4615, -1.6574],
        [-0.0936, -0.1915,  0.2088, -0.8390],
        [-1.9461, -2.5019,  1.6047, -0.2060],
        [-0.8711,  0.5696,  1.3668,  1.2347],
        [ 0.0842, -0.3537, -0.0246, -0.7429]], grad_fn=<EmbeddingBackward0>)


In [57]:
# 如何进行压缩？->扩建词表->将原本只表示0~255的token扩展到更大的范围。
Dict = {}
for key in range(0,256):
    value = key
    Dict[key] = value
print("原始词表大小：", len(Dict)) # 256,范围是0~255
print("原始词表内容样例：{key}: {value}".format(key=0, value=Dict[0])) # 0: 0

# 词表扩展样例，例如：{256：239，164}，表示将原本的两个token 239和164合并成一个新的token 256。
Dict[256] = [228, 184]
Dict[257] = [173, 230, 156, 172]
Dict[6666] = [256, 173, 257]
print("扩展后的词表大小：", len(Dict)) # 257,范围是0~256
print("扩展后的词表内容样例：{key}: {value}".format(key=256, value=Dict[256])) # 256: [228, 184]
print("扩展后的词表内容样例：{key}: {value}".format(key=257, value=Dict[257])) # 257: [230, 156, 172]
# 此时，大模型的token就可以从[228, 184, 173, 230, 156, 172]压缩成[256, 173, 257],以此类推，最终"大模型"就可以压缩成[6666]，用一个token表示。
X_1 = nn.Embedding(6667, 4) # 6667个token，每个token对应一个4维的向量
idx_1 = torch.tensor([6666]) # "大模型"的每个数字都是对应的vector的索引，转成Embedding可以识别的类型
print("随机初始化的，所以每一次的值都不同:\n" + str(X_1(idx_1))) # 将"大模型"用最终的[ 0.7202,  1.6928, -0.4975, -1.6373]表示

print("====================================================================================\n" + 
'''
上述合并以及扩建词表的方法就是BPE。核心目标：压缩原始输入的token数量，减少序列长度，降低Transformer自注意力O(N²)的计算开销。
''')

print('''
<!-- 特殊整词Token（子词补充策略） -->
高频完整词语直接作为独立Token存入词表，跳过字节拆分，进一步压缩输入Token总数。
原理：类似BPE合并的终极形态，提前把高频短语注册为单独词表条目。
示例词表映射：
Dict[6667] = ["大模型"]
Dict[8888] = ["心脏病"]
<!-- 双重收益 -->
1. 扩充词表覆盖范围，避免生僻高频词完全拆分为零散字节；
2. 原本"大模型"会拆成多个UTF-8字节Token，使用整词Token后仅占用1个ID，大幅缩短序列长度，降低自注意力O(N²)计算量。
''')


原始词表大小： 256
原始词表内容样例：0: 0
扩展后的词表大小： 259
扩展后的词表内容样例：256: [228, 184]
扩展后的词表内容样例：257: [173, 230, 156, 172]
随机初始化的，所以每一次的值都不同:
tensor([[ 0.7427,  1.1428,  0.2900, -1.0807]], grad_fn=<EmbeddingBackward0>)

上述合并以及扩建词表的方法就是BPE。核心目标：压缩原始输入的token数量，减少序列长度，降低Transformer自注意力O(N²)的计算开销。


<!-- 特殊整词Token（子词补充策略） -->
高频完整词语直接作为独立Token存入词表，跳过字节拆分，进一步压缩输入Token总数。
原理：类似BPE合并的终极形态，提前把高频短语注册为单独词表条目。
示例词表映射：
Dict[6667] = ["大模型"]
Dict[8888] = ["心脏病"]
<!-- 双重收益 -->
1. 扩充词表覆盖范围，避免生僻高频词完全拆分为零散字节；
2. 原本"大模型"会拆成多个UTF-8字节Token，使用整词Token后仅占用1个ID，大幅缩短序列长度，降低自注意力O(N²)计算量。

